# Training on pre-patched data

Instead of sampling patches from large rasters during training, you can split your data into patch files once, with one patch per GeoTIFF, and train on those. This is useful when you want full control over which patches are used, e.g. to skip mostly empty areas, or when your data already comes as patches.

The patch files are loaded as a regular eotorch dataset, so plotting, the map preview and the splits work the same way as for large rasters.

In [ ]:
import shutil
from pathlib import Path

from lightning import seed_everything
from lightning.pytorch import Trainer

from eotorch.data import (
    RasterDataModule,
    SegmentationTask,
    get_segmentation_dataset,
)
from eotorch.processing import generate_patches_from_files

# class mapping, for plotting and clarity only, not required
class_mapping = {
    1: "Optically deep water",
    2: "Sand",
    3: "Coral",
    4: "SAV",
}
DATASET_DIR = Path("tests/data")
PATCH_DIR = Path("patches")
PATCH_SIZE = 32

### Create the patch files

'generate_patches_from_files' splits an image and its labels into patches of 'patch_size' pixels. A random 'val_fraction' of the patches is written to a 'val' directory, the rest to a 'train' directory. With 'n_random_offsets', extra training patches shifted by up to half a patch are added for more variety. These never overlap the validation patches.

Patches with too many nodata pixels are skipped (see 'empty_img_threshold', 'empty_label_threshold' and 'value_threshold'). The labels may differ from the image in extent, resolution and CRS, in which case they are reprojected onto the image grid.

If you bring your own patches instead, make sure that:
- each patch is a georeferenced GeoTIFF, with the image and the label in separate files
- the image and label file of a patch have exactly the same extent
- all patches have the same size

Here, we use three of the test images for training and validation, and keep the fourth for predicting.

In [ ]:
shutil.rmtree(PATCH_DIR, ignore_errors=True)  # start from scratch when re-running
for i in range(3):
    generate_patches_from_files(
        img_path=DATASET_DIR / f"image_{i}.tif",
        label_path=DATASET_DIR / f"label_{i}.tif",
        out_dir=PATCH_DIR,
        patch_size=PATCH_SIZE,
        val_fraction=0.2,
        n_random_offsets=2,
    )

for split in ("train", "val"):
    n = len(list((PATCH_DIR / split).glob("*_image.tif")))
    print(f"{split}: {n} patches")

### Create the datasets and visualize some samples

The patch directories are loaded with the same function as large rasters. The glob patterns tell image and label files apart, and each image patch is matched with the label patch at the same location.

In [ ]:
dataset_args = dict(
    image_glob="*_image.tif",
    label_glob="*_label.tif",
    all_image_bands=("red", "green", "blue"),
    rgb_bands=("red", "green", "blue"),
    class_mapping=class_mapping,
    reduce_zero_label=True,
)
train_ds = get_segmentation_dataset(
    images_dir=PATCH_DIR / "train", labels_dir=PATCH_DIR / "train", **dataset_args
)
val_ds = get_segmentation_dataset(
    images_dir=PATCH_DIR / "val", labels_dir=PATCH_DIR / "val", **dataset_args
)
train_ds.plot_samples(n=3, patch_size=PATCH_SIZE)

### Create a data module for training

Setting 'patch_size=None' tells the data module that the datasets are already split into patches: every patch file is one sample, and the patch size is read from the files. Each sample is read from its own file only, so overlapping patches are not merged with each other.

The images are normalized on the fly with the per-band mean and standard deviation of the training patches. These are saved in the checkpoint, so the same normalization is applied when predicting.

In [ ]:
from eotorch.processing.normalize import band_statistics

seed_everything(43, workers=False)

mean, std = band_statistics(train_ds)
module = RasterDataModule(
    train_dataset=train_ds,
    val_dataset=val_ds,
    batch_size=8,
    patch_size=None,
    mean=mean,
    std=std,
    # num_workers=6,
    # persistent_workers=True,
)

Preview the patches of each dataset on an interactive map, to check that the training and validation patches don't overlap.

In [ ]:
module.preview_data_sampling(max_samples=None)

### Create a task and a trainer

The task and the trainer are set up exactly as for training on large rasters.

In [ ]:
from lightning.pytorch.callbacks import ModelCheckpoint

task = SegmentationTask(
    model="deepresunet",
    in_channels=3,
    num_classes=len(class_mapping),
    ignore_index=-1,  # because reduce_zero_label=True
)
trainer = Trainer(
    callbacks=[
        ModelCheckpoint(
            monitor="val/F1_Score",
            mode="max",
            filename="{epoch}-{val/F1_Score:.2f}",
        ),
    ],
    max_epochs=30,
)

In [ ]:
trainer.fit(model=task, datamodule=module)

### Use your model to make predictions

Predictions are made on full rasters, not on patches. The patch size and the normalization are read from the checkpoint, and with 'blend' set to True, overlapping patch predictions are blended to avoid visible seams.

In [ ]:
f = trainer.checkpoint_callback.best_model_path
print(f"Using weights {f}")
SegmentationTask.predict_on_tif_file(
    tif_file_path=DATASET_DIR / "image_3.tif",
    checkpoint_path=f,
    out_file_path="predictions/image_3_pred.tif",
    show_results=True,
    blend=True,
)